# Aggregations and Grouping with Dataframes

Using the F1 source data files in the folder "f1-sourcefiles/" create DataFrames.

This jupyter notebook covers:
- `groupBy()` + `agg()` with functions like `sum()`, `count()`, `avg()`, `max()`, `min()`
- `pivot()` for reshaping data
- Window functions (`ROW_NUMBER`, `RANK`, `LAG`, `LEAD`)


In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/03/30 21:42:09 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/30 21:42:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dae3d134-04f5-4911-a3bc-e22b9ea884ec;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 306ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

In [2]:
# Load drivers.csv file
import time

start = time.time()

df_drivers = spark.read.csv("f1-sourcefiles/drivers.csv", header=True, inferSchema=True)

df_drivers.show()

end = time.time()

print(f"Time takes: {end - start:.2f} seconds")


26/03/26 23:14:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+----------+------+----+---------+----------+----------+-----------+--------------------+
|driverId| driverRef|number|code| forename|   surname|       dob|nationality|                 url|
+--------+----------+------+----+---------+----------+----------+-----------+--------------------+
|       1|  hamilton|    44| HAM|    Lewis|  Hamilton|1985-01-07|    British|http://en.wikiped...|
|       2|  heidfeld|    \N| HEI|     Nick|  Heidfeld|1977-05-10|     German|http://en.wikiped...|
|       3|   rosberg|     6| ROS|     Nico|   Rosberg|1985-06-27|     German|http://en.wikiped...|
|       4|    alonso|    14| ALO| Fernando|    Alonso|1981-07-29|    Spanish|http://en.wikiped...|
|       5|kovalainen|    \N| KOV|   Heikki|Kovalainen|1981-10-19|    Finnish|http://en.wikiped...|
|       6|  nakajima|    \N| NAK|   Kazuki|  Nakajima|1985-01-11|   Japanese|http://en.wikiped...|
|       7|  bourdais|    \N| BOU|Sébastien|  Bourdais|1979-02-28|     French|http://en.wikiped...|
|       8|

In [3]:
# GroupBy() 
df_drivers.groupBy("nationality").count().show()


+-----------------+-----+
|      nationality|count|
+-----------------+-----+
|          Mexican|    6|
|        Rhodesian|    4|
|          Finnish|    9|
|            Swiss|   23|
|             Thai|    2|
|           Indian|    2|
|    South African|   23|
|          Chinese|    1|
|       Indonesian|    1|
|Argentine-Italian|    1|
|          Chilean|    1|
|            Irish|    5|
|        Argentine|   24|
|           Polish|    1|
|          British|  166|
|         Japanese|   20|
|       Australian|   19|
|          Spanish|   15|
|       Portuguese|    4|
|        Hungarian|    1|
+-----------------+-----+
only showing top 20 rows



In [3]:
# .groupBy() - agg() with mean(), sum() count(), min() and max() by driver


from pyspark.sql import functions as F

df_results = spark.read.csv("f1-sourcefiles/results.csv", header=True, inferSchema=True)

df_results.groupBy("driverId").agg(
    F.min("positionOrder").alias("min_final_postiion"),
    F.max("positionOrder").alias("max_final_postiion"),
    F.mean("positionOrder").alias("mean_final_postiion"),
    F.sum("laps").alias("total_laps"),
    F.count("*").alias("total_races")
).show()


+--------+------------------+------------------+-------------------+----------+-----------+
|driverId|min_final_postiion|max_final_postiion|mean_final_postiion|total_laps|total_races|
+--------+------------------+------------------+-------------------+----------+-----------+
|     148|                 7|                38| 27.636363636363637|       299|         22|
|     463|                15|                29| 24.333333333333332|        64|          3|
|     471|                 9|                 9|                9.0|        76|          1|
|     496|                 3|                21|               10.5|       541|         12|
|     833|                12|                19| 15.692307692307692|       737|         13|
|     243|                 3|                33| 14.721311475409836|      2452|         61|
|     392|                11|                19|               15.0|        81|          2|
|     540|                19|                23|               21.0|        17| 

In [12]:
# Groupby() multiple columns


df_results.groupBy("driverId", "constructorId").agg(
    F.min("positionOrder").alias("min_final_postiion"),
    F.max("positionOrder").alias("max_final_postiion"),
    F.mean("positionOrder").alias("mean_final_postiion"),
    F.sum("laps").alias("total_laps"),
    F.count("*").alias("total_races")
).show()


+--------+-------------+------------------+------------------+-------------------+----------+-----------+
|driverId|constructorId|min_final_postiion|max_final_postiion|mean_final_postiion|total_laps|total_races|
+--------+-------------+------------------+------------------+-------------------+----------+-----------+
|      43|            7|                 6|                20| 12.107142857142858|      1484|         28|
|     158|           27|                 2|                29|               13.8|       648|         15|
|     408|           66|                 4|                23|               13.6|       154|          5|
|     485|          172|                28|                28|               28.0|         0|          1|
|     478|           66|                 9|                 9|                9.0|        36|          1|
|     554|          105|                 2|                23|               7.96|      1390|         25|
|     459|           87|                 8|   

In [8]:
# groupBy() with a pivot() function

# ── Sample data ───────────────────────────────────────────────────────────────
data = [
    ("Apple",  "2022", 100),
    ("Apple",  "2023", 130),
    ("Apple",  "2024", 160),
    ("Banana", "2022",  80),
    ("Banana", "2023",  95),
    ("Banana", "2024", 110),
    ("Cherry", "2022",  50),
    ("Cherry", "2023",  70),
    ("Cherry", "2024",  90),
]

df = spark.createDataFrame(data, ["product", "year", "sales"])
 
print("INPUT — long format")
df.show()


# ── 1. Basic pivot ────────────────────────────────────────────────────────────
# Each distinct value in 'year' becomes a new column.
# The cell value is sum("sales") for that product/year combination.
# Passing the values explicitly ["2022","2023","2024"] is faster —
# without them Spark runs an extra job to discover distinct values.
 
print("1. Basic pivot — total sales per product per year")
df.groupBy("product") \
  .pivot("year", ["2022", "2023", "2024"]) \
  .sum("sales") \
  .show()
# +-------+----+----+----+
# |product|2022|2023|2024|
# +-------+----+----+----+
# |  Apple| 100| 130| 160|
# | Banana|  80|  95| 110|
# | Cherry|  50|  70|  90|
# +-------+----+----+----+

# ── 2. Multiple aggregations ──────────────────────────────────────────────────
# .agg() lets you compute more than one metric per pivot value.
# Column names are auto-generated as <pivot_value>_<alias>.
 
print("2. Multi-agg pivot — sum and avg per year")
df.groupBy("product") \
  .pivot("year", ["2022", "2023", "2024"]) \
  .agg(
      F.sum("sales").alias("sum"),
      F.round(F.avg("sales"), 1).alias("avg"),
  ) \
  .show()
# +-------+--------+--------+--------+--------+--------+--------+
# |product|2022_sum|2022_avg|2023_sum|2023_avg|2024_sum|2024_avg|
# +-------+--------+--------+--------+--------+--------+--------+
# |  Apple|     100|   100.0|     130|   130.0|     160|   160.0|
# ...


# ── 3. Dynamic pivot — values discovered at runtime ───────────────────────────
# Useful when you don't know the pivot values in advance.
 
print("3. Dynamic pivot — years discovered from data")
pivot_vals = (
    df.select("year")
      .distinct()
      .orderBy("year")
      .rdd.flatMap(lambda x: x)
      .collect()
)
df.groupBy("product") \
  .pivot("year", pivot_vals) \
  .sum("sales") \
  .show()


# ── 4. Unpivot — reverse a pivot (wide → long) ────────────────────────────────
# stack(N, 'label1', col1, 'label2', col2, ...) reshapes N wide columns
# back into two columns: a label column and a value column.
 
print("4. Unpivot — wide back to long using stack()")
wide = df.groupBy("product") \
         .pivot("year", ["2022", "2023", "2024"]) \
         .sum("sales")
 
wide.selectExpr(
    "product",
    "stack(3, '2022',`2022`, '2023',`2023`, '2024',`2024`) as (year, sales)"
).orderBy("product", "year").show()
# Produces the original long-format table again.
 
# PySpark 3.4+ also has a native unpivot() method:
# wide.unpivot(
#     ids=["product"],
#     values=["2022", "2023", "2024"],
#     variableColumnName="year",
#     valueColumnName="sales",
# )
 
spark.stop()


INPUT — long format
+-------+----+-----+
|product|year|sales|
+-------+----+-----+
|  Apple|2022|  100|
|  Apple|2023|  130|
|  Apple|2024|  160|
| Banana|2022|   80|
| Banana|2023|   95|
| Banana|2024|  110|
| Cherry|2022|   50|
| Cherry|2023|   70|
| Cherry|2024|   90|
+-------+----+-----+

1. Basic pivot — total sales per product per year
+-------+----+----+----+
|product|2022|2023|2024|
+-------+----+----+----+
| Banana|  80|  95| 110|
| Cherry|  50|  70|  90|
|  Apple| 100| 130| 160|
+-------+----+----+----+

2. Multi-agg pivot — sum and avg per year
+-------+--------+--------+--------+--------+--------+--------+
|product|2022_sum|2022_avg|2023_sum|2023_avg|2024_sum|2024_avg|
+-------+--------+--------+--------+--------+--------+--------+
| Banana|      80|    80.0|      95|    95.0|     110|   110.0|
| Cherry|      50|    50.0|      70|    70.0|      90|    90.0|
|  Apple|     100|   100.0|     130|   130.0|     160|   160.0|
+-------+--------+--------+--------+--------+--------+

# Windowing functions - (ROW_NUMBER, RANK, LAG, LEAD)



# ROW_NUMBER in PySpark

Here's a full explanation of ROW_NUMBER in PySpark — one of the most useful window functions for ranking rows within groups.

What it does: ROW_NUMBER() assigns a unique sequential integer to each row within a partition (group), ordered by a column you specify. Unlike RANK() or DENSE_RANK(), it never produces ties — every row gets a distinct number.

The core syntax always involves three things: the ROW_NUMBER() function, a Window spec defining how to group (partitionBy) and sort (orderBy), and .withColumn() to attach the result.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("row_number_demo").getOrCreate()

# Sample sales data
data = [
    ("Alice", "Electronics", 1200),
    ("Bob",   "Electronics",  950),
    ("Carol", "Electronics", 1100),
    ("Dave",  "Clothing",     400),
    ("Eve",   "Clothing",     600),
    ("Frank", "Clothing",     400),
    ("Frank", "Clothing",     500),
    ("Frank", "Clothing",     600),
]
df = spark.createDataFrame(data, ["name", "category", "sales"])

# Define the window: group by category, rank by sales descending
window_spec = Window.partitionBy("category").orderBy(F.asc("sales"))

# Apply ROW_NUMBER
df_ranked = df.withColumn("row_num", F.row_number().over(window_spec))
df_ranked.show()

# ```

# Output:

# +-----+-----------+-----+-------+
# | name|   category|sales|row_num|
# +-----+-----------+-----+-------+
# |Alice|Electronics| 1200|      1|
# |Carol|Electronics| 1100|      2|
# |  Bob|Electronics|  950|      3|
# |  Eve|    Clothing|  600|      1|
# | Dave|    Clothing|  400|      2|  ← ties get arbitrary but distinct numbers
# |Frank|    Clothing|  400|      3|
# +-----+-----------+-----+-------+

# ```

# Notice that Dave and Frank both have sales = 400 but get different row numbers (2 and 3). That's the key difference from RANK().

26/04/09 22:55:31 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/09 22:55:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 22:55:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-----+-----------+-----+-------+
| name|   category|sales|row_num|
+-----+-----------+-----+-------+
| Dave|   Clothing|  400|      1|
|Frank|   Clothing|  400|      2|
|Frank|   Clothing|  500|      3|
|  Eve|   Clothing|  600|      4|
|Frank|   Clothing|  600|      5|
|  Bob|Electronics|  950|      1|
|Carol|Electronics| 1100|      2|
|Alice|Electronics| 1200|      3|
+-----+-----------+-----+-------+



26/04/09 22:55:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [8]:
# The most common use case — getting the top-N per group:

# Top 2 sellers per category
top2 = df_ranked.filter(F.col("row_num") <= 2)
top2.show()


+-----+-----------+-----+-------+
| name|   category|sales|row_num|
+-----+-----------+-----+-------+
| Dave|   Clothing|  400|      1|
|Frank|   Clothing|  400|      2|
|  Bob|Electronics|  950|      1|
|Carol|Electronics| 1100|      2|
+-----+-----------+-----+-------+



In [6]:
# Deduplication — keep one row per key:

# If you have duplicates and want only the latest record per name
window_spec = Window.partitionBy("name").orderBy(F.desc("sales"))
df_top = df.withColumn("row_number", F.row_number().over(window_spec)) \
             .filter(F.col("row_number") == 1) # \
             # .drop("row_number")

df_top.show()


+-----+-----------+-----+----------+
| name|   category|sales|row_number|
+-----+-----------+-----+----------+
|Alice|Electronics| 1200|         1|
|  Bob|Electronics|  950|         1|
|Carol|Electronics| 1100|         1|
| Dave|   Clothing|  400|         1|
|  Eve|   Clothing|  600|         1|
|Frank|   Clothing|  600|         1|
+-----+-----------+-----+----------+



# How PySpark RANK works

**Key things to remember about rank():**

**Takes no arguments.** Unlike lag, rank() needs no column name — the ordering is entirely controlled by orderBy in the window spec.

**The tie + gap rule is the defining characteristic.** When two rows are tied, they share a rank and the next rank is skipped. So if two rows tie at rank 2, the next rank is 4 — there's no rank 3. This mirrors how sports standings work (two silver medals, no bronze).

**Know when to use which sibling function:**

- rank() — ties share rank, gaps follow. Use for leaderboards where ties are meaningful.
- dense_rank() — ties share rank, no gaps. Use when you care about the ordinal position (1st, 2nd, 3rd) not the count of rows ahead.
- row_number() — always unique, no ties. Use when you need exactly N rows (e.g. strictly top-1 per group).

**Top-N filtering gotcha.** Filtering rank <= 2 can return more than 2 rows per partition if there are ties at the boundary. If you need a strict row count, switch to row_number().

The pipeline deliberately includes ties (Bob & Carol at 95k, Frank & Grace at 75k) to show that rank() == 1 still returns a clean single winner per department when the tie isn't at the top. If there were two people both earning the max salary in the same department, rank() == 1 would return both of them — which is often exactly what you want (include all joint winners), but use row_number() == 1 if you ever need to force exactly one row.

The .drop("rank") at the end is a common pattern — compute the rank as a helper column to filter on, then remove it so the output stays clean.


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

data = [
    ("Engineering", "Alice",   120000),
    ("Engineering", "Bob",     95000),
    ("Engineering", "Carol",   95000),  # tie!
    ("Engineering", "Dan",     80000),
    ("Sales",       "Eve",     90000),
    ("Sales",       "Frank",   75000),
    ("Sales",       "Grace",   75000),  # tie!
    ("HR",          "Henry",   70000),
    ("HR",          "Isla",    65000),
]

df = spark.createDataFrame(data, ["dept", "name", "salary"])

df.show()

+-----------+-----+------+
|       dept| name|salary|
+-----------+-----+------+
|Engineering|Alice|120000|
|Engineering|  Bob| 95000|
|Engineering|Carol| 95000|
|Engineering|  Dan| 80000|
|      Sales|  Eve| 90000|
|      Sales|Frank| 75000|
|      Sales|Grace| 75000|
|         HR|Henry| 70000|
|         HR| Isla| 65000|
+-----------+-----+------+



In [9]:
# Define the window: group by dept, order by salary descending
window_spec = (
    Window
    .partitionBy("dept")
    .orderBy(F.desc("salary"))
)

In [13]:
# Add the rank column
ranked_df = df.withColumn(
    "rank",
    F.rank().over(window_spec)
)

ranked_df.show()

+-----------+-----+------+----+
|       dept| name|salary|rank|
+-----------+-----+------+----+
|Engineering|Alice|120000|   1|
|Engineering|  Bob| 95000|   2|
|Engineering|Carol| 95000|   2|
|Engineering|  Dan| 80000|   4|
|         HR|Henry| 70000|   1|
|         HR| Isla| 65000|   2|
|      Sales|  Eve| 90000|   1|
|      Sales|Frank| 75000|   2|
|      Sales|Grace| 75000|   2|
+-----------+-----+------+----+



In [20]:
# Filter to keep only rank 1 — the top earner(s) per dept
top_earners = ranked_df.filter(F.col("rank") == 1).drop("rank")

top_earners.show()

+-----------+-----+------+
|       dept| name|salary|
+-----------+-----+------+
|Engineering|Alice|120000|
|         HR|Henry| 70000|
|      Sales|  Eve| 90000|
+-----------+-----+------+

